In [1]:
import numpy as np #helps us in numerical operations
import pandas as pd # helps us to work with tables
import matplotlib.pyplot as plt #helps us to plot some graphs, simple plots


# Scikit-learn tools
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.ensemble import IsolationForest

from IPython.display import display

# Display settings for better table viewing
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:

## Load and Read credit_applicants.csv file

df_txn_behaviour = pd.read_csv("data/txn_behaviour.csv")

# =========================
# 2. Basic inspection (No. of rows x columns)
# =========================
print("Credit applicant data size: ", df_txn_behaviour.shape)



Credit applicant data size:  (265, 6)


In [3]:
numeric_behavioural_features = ["txn_hour", "is_new_device", "txn_amount_inr"]



# 2. Standardize the numerical columns
# standardizing is best practice in analytical pipelines for uniform handling.
scaler = StandardScaler()
df_scaled_features = scaler.fit_transform(df_txn_behaviour[["txn_hour", "is_new_device", "txn_amount_inr"]])


#print(scaled_features)

In [4]:


## Given the contamination rate = 5.7%
injected_anomaly_proportion = 5.7
contamination_rate = injected_anomaly_proportion /100




#3. Fit Isolation Forest matching the contamination rate
# Passing the calculated proportion guarantees it flags the exact right subset volume.
iso_forest = IsolationForest(contamination=contamination_rate, random_state=42)


# Create a df_txn_behaviour copy
df_pred = df_txn_behaviour.copy()


# Predictions (-1 for anomalies, 1 for inliers)
df_pred['Raw_Pred'] = iso_forest.fit_predict(df_scaled_features)

# 4. Map predictions to standard binary labels (0 = Normal, 1 = Anomaly)
df_pred['Predicted_Anomaly'] = df_pred['Raw_Pred'].apply(lambda x: 1 if x == -1 else 0)


# 5. Evaluate predictions
print("\nPredicted Anomaly Count Breakdown:")
print(df_pred['Predicted_Anomaly'].value_counts())


## Print df_pred where df_pred['Predicted_Anomaly'] == 1
print("\nPredicted Anomalies:")
print(df_pred[df_pred['Predicted_Anomaly'] == 1])





Predicted Anomaly Count Breakdown:
Predicted_Anomaly
0    249
1     16
Name: count, dtype: int64

Predicted Anomalies:
       txn_id applicant_id  txn_hour  is_new_device  txn_amount_inr channel  Raw_Pred  Predicted_Anomaly
3    BTXN5003      APP1293        18              1             999     P2M        -1                  1
10   BTXN5010      APP1027        22              1             199     P2P        -1                  1
81   BTXN5081      APP1354        13              1            3999     P2P        -1                  1
130  BTXN5130      APP1328         7              1            1999     P2M        -1                  1
157  BTXN5157      APP1108        20              1             999     P2P        -1                  1
250    BTXNA0      APP1089         2              1           24999     P2P        -1                  1
253    BTXNA3      APP1357         4              1           19999     P2P        -1                  1
254    BTXNA4      APP1173         4    

In [5]:
# 5. Evaluate predictions
print("\nPredicted Anomaly Count Breakdown:")
print(df_pred['Predicted_Anomaly'].value_counts())


## Print df_pred where df_pred['Predicted_Anomaly'] == 1
print("\nPredicted Anomalies:")
print(df_pred[df_pred['Predicted_Anomaly'] == 1])


Predicted Anomaly Count Breakdown:
Predicted_Anomaly
0    249
1     16
Name: count, dtype: int64

Predicted Anomalies:
       txn_id applicant_id  txn_hour  is_new_device  txn_amount_inr channel  Raw_Pred  Predicted_Anomaly
3    BTXN5003      APP1293        18              1             999     P2M        -1                  1
10   BTXN5010      APP1027        22              1             199     P2P        -1                  1
81   BTXN5081      APP1354        13              1            3999     P2P        -1                  1
130  BTXN5130      APP1328         7              1            1999     P2M        -1                  1
157  BTXN5157      APP1108        20              1             999     P2P        -1                  1
250    BTXNA0      APP1089         2              1           24999     P2P        -1                  1
253    BTXNA3      APP1357         4              1           19999     P2P        -1                  1
254    BTXNA4      APP1173         4    